In [2]:
!pip install --upgrade pip
!pip install langchain langchain-community transformers --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 29.5 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [3]:
#Import + tạo factory cho ConversationChain
from transformers import pipeline
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferWindowMemory
from langchain_community.llms import HuggingFacePipeline

def create_conversation_chain(k_value, model_name="bigscience/mt0-base", max_length=256):
    generator = pipeline(
        "text2text-generation",
        model=model_name,
        max_length=max_length
    )
    llm = HuggingFacePipeline(pipeline=generator)
    memory = ConversationBufferWindowMemory(k=k_value)  # giữ k lượt gần nhất
    conv = ConversationChain(llm=llm, memory=memory, verbose=False)
    return conv

In [4]:
system_prompt = (
    "Bạn là một trợ lý thân thiện và ngắn gọn. "
    "Trả lời rõ ràng bằng tiếng Việt. Nếu không biết, nói 'Tôi không biết'. "
    "Luôn lịch sự, đừng thêm thông tin không có trong ngữ cảnh."
)

# Bộ câu hỏi cố định
inputs = [
    "Chào bạn.",
    "Tôi thích màu xanh.",
    "Tôi sống ở Hà Nội.",
    "Bạn nhớ tôi thích màu gì không?"
]

print("System prompt:\n", system_prompt)
print("\nCâu hỏi mẫu sẽ được dùng (theo thứ tự):")
for i, s in enumerate(inputs, 1):
    print(i, s)

System prompt:
 Bạn là một trợ lý thân thiện và ngắn gọn. Trả lời rõ ràng bằng tiếng Việt. Nếu không biết, nói 'Tôi không biết'. Luôn lịch sự, đừng thêm thông tin không có trong ngữ cảnh.

Câu hỏi mẫu sẽ được dùng (theo thứ tự):
1 Chào bạn.
2 Tôi thích màu xanh.
3 Tôi sống ở Hà Nội.
4 Bạn nhớ tôi thích màu gì không?


In [7]:
for k in [1, 2, 3]:
    print("\n" + "="*80)
    print(f"🔁 THỬ NGHIỆM VỚI k = {k}")
    print("="*80)
    conv = create_conversation_chain(k_value=k, model_name="bigscience/mt0-base", max_length=256)

    for turn_idx, user_msg in enumerate(inputs, start=1):
        combined_input = f"{system_prompt}\n\nUser: {user_msg}\nAssistant:"
        response = conv.run(combined_input)

        print(f"\nTurn {turn_idx} - User: {user_msg}")
        print(f"Bot: {response}\n")

        # In trạng thái memory (buffer) sau mỗi lượt
        print(f"🧠 Memory (buffer) after turn {turn_idx} (k={k}):")
        try:
            print(conv.memory.buffer)
        except Exception as e:
            print(f"(Không in được conv.memory.buffer: {e})")
        print("-"*60)

    print("\nKẾT THÚC thử nghiệm k =", k)
    print("Memory cuối cùng (raw):")
    print(conv.memory.buffer)
    print("\n\n")



🔁 THỬ NGHIỆM VỚI k = 1


Device set to use cpu
/tmp/ipython-input-2504337096.py:9: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = conv.run(combined_input)



Turn 1 - User: Chào bạn.
Bot: Human: Bạn là một trợ lý thân thiện và ngắn gọn. Trả lời rõ ràng bằng tiếng Việt.

🧠 Memory (buffer) after turn 1 (k=1):
Human: Bạn là một trợ lý thân thiện và ngắn gọn. Trả lời rõ ràng bằng tiếng Việt. Nếu không biết, nói 'Tôi không biết'. Luôn lịch sự, đừng thêm thông tin không có trong ngữ cảnh.

User: Chào bạn.
Assistant:
AI: Human: Bạn là một trợ lý thân thiện và ngắn gọn. Trả lời rõ ràng bằng tiếng Việt.
------------------------------------------------------------

Turn 2 - User: Tôi thích màu xanh.
Bot: Human: Bạn là một trợ lý thân thiện và ngắn gọn. Trả lời rõ ràng bằng tiếng Việt. Nếu không biết, nói 'Tôi không biết'. Luôn lịch sự, đừng thêm thông tin không có trong ngữ cảnh.

🧠 Memory (buffer) after turn 2 (k=1):
Human: Bạn là một trợ lý thân thiện và ngắn gọn. Trả lời rõ ràng bằng tiếng Việt. Nếu không biết, nói 'Tôi không biết'. Luôn lịch sự, đừng thêm thông tin không có trong ngữ cảnh.

User: Tôi thích màu xanh.
Assistant:
AI: Human: Bạn là 

Device set to use cpu



Turn 1 - User: Chào bạn.
Bot: Human: Bạn là một trợ lý thân thiện và ngắn gọn. Trả lời rõ ràng bằng tiếng Việt.

🧠 Memory (buffer) after turn 1 (k=2):
Human: Bạn là một trợ lý thân thiện và ngắn gọn. Trả lời rõ ràng bằng tiếng Việt. Nếu không biết, nói 'Tôi không biết'. Luôn lịch sự, đừng thêm thông tin không có trong ngữ cảnh.

User: Chào bạn.
Assistant:
AI: Human: Bạn là một trợ lý thân thiện và ngắn gọn. Trả lời rõ ràng bằng tiếng Việt.
------------------------------------------------------------

Turn 2 - User: Tôi thích màu xanh.
Bot: Human: Bạn là một trợ lý thân thiện và ngắn gọn. Trả lời rõ ràng bằng tiếng Việt. Nếu không biết, nói 'Tôi không biết'. Luôn lịch sự, đừng thêm thông tin không có trong ngữ cảnh.

🧠 Memory (buffer) after turn 2 (k=2):
Human: Bạn là một trợ lý thân thiện và ngắn gọn. Trả lời rõ ràng bằng tiếng Việt. Nếu không biết, nói 'Tôi không biết'. Luôn lịch sự, đừng thêm thông tin không có trong ngữ cảnh.

User: Chào bạn.
Assistant:
AI: Human: Bạn là một trợ lý

Device set to use cpu



Turn 1 - User: Chào bạn.
Bot: Human: Bạn là một trợ lý thân thiện và ngắn gọn. Trả lời rõ ràng bằng tiếng Việt.

🧠 Memory (buffer) after turn 1 (k=3):
Human: Bạn là một trợ lý thân thiện và ngắn gọn. Trả lời rõ ràng bằng tiếng Việt. Nếu không biết, nói 'Tôi không biết'. Luôn lịch sự, đừng thêm thông tin không có trong ngữ cảnh.

User: Chào bạn.
Assistant:
AI: Human: Bạn là một trợ lý thân thiện và ngắn gọn. Trả lời rõ ràng bằng tiếng Việt.
------------------------------------------------------------

Turn 2 - User: Tôi thích màu xanh.
Bot: Human: Bạn là một trợ lý thân thiện và ngắn gọn. Trả lời rõ ràng bằng tiếng Việt. Nếu không biết, nói 'Tôi không biết'. Luôn lịch sự, đừng thêm thông tin không có trong ngữ cảnh.

🧠 Memory (buffer) after turn 2 (k=3):
Human: Bạn là một trợ lý thân thiện và ngắn gọn. Trả lời rõ ràng bằng tiếng Việt. Nếu không biết, nói 'Tôi không biết'. Luôn lịch sự, đừng thêm thông tin không có trong ngữ cảnh.

User: Chào bạn.
Assistant:
AI: Human: Bạn là một trợ lý